# KidsNutriBite: Google Colab Setup and Evaluation Playground

Welcome to the Google Colab playbook for the KidsNutriBite research prototype. This notebook is designed to guide you through setting up the environment, verifying model connections, indexing the RAG dataset, performing semantic searches, and executing model evaluations.

## Step 1: Install Dependencies
Run the following cell to install the required libraries on the Colab environment.

In [ ]:
!pip install sentence-transformers faiss-cpu google-generativeai pandas tabulate matplotlib groq transformers accelerate bitsandbytes

## Step 2: Configure API Keys
Enter your Google Gemini API key below. The Gemini key is required to run the evaluation judge.

In [ ]:
import os

# Handle Kaggle environment automatically
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        os.environ['GEMINI_API_KEY'] = user_secrets.get_secret('GEMINI_API_KEY')
        os.environ['GROQ_API_KEY'] = user_secrets.get_secret('GROQ_API_KEY')
        print('Loaded API keys from Kaggle Secrets.')
    except Exception as e:
        print('Could not load Kaggle secrets:', e)
else:
    # Set your API Keys here for Colab
    os.environ['GEMINI_API_KEY'] = 'YOUR_GEMINI_API_KEY_HERE'
    os.environ['GROQ_API_KEY'] = 'YOUR_GROQ_API_KEY_HERE'

# Write keys to .env for the CLI orchestrator
with open('.env', 'w') as f:
    if os.environ.get('GEMINI_API_KEY'):
        f.write(f'GEMINI_API_KEY="{os.environ["GEMINI_API_KEY"]}"\n')
    if os.environ.get('GROQ_API_KEY'):
        f.write(f'GROQ_API_KEY="{os.environ["GROQ_API_KEY"]}"\n')

print('Environment keys configured successfully!')

## Step 3: Run Model Connection Diagnostics
Run the diagnostic scripts to verify that the Gemini API is reachable and that local open-source models can run successfully in the GPU-accelerated environment.

In [ ]:
print("=== Running Gemini Diagnostics ===")
!python verify_gemini.py

print("\n=== Running Groq Diagnostics ===")
!python verify_groq.py

print("\n=== Running Qwen Local Diagnostics ===")
!python verify_qwen.py

## Step 4: Run RAG Indexing
Build the FAISS semantic vector database index using the SentenceTransformer `BAAI/bge-small-en-v1.5` model.

In [ ]:
!python main.py --index

## Step 5: Run Semantic RAG Search (Retriever Test)
Verify that retrieval queries successfully retrieve the relevant nutritional rules and clinical rules.

In [ ]:
!python main.py --query "What is the feeding protocol for an infant with diarrhea?"


## Step 6: Test Diet Planner & QA Response
Generate a complete meal plan and ask the Gemini model to explain it under safety constraints.

In [ ]:
!python main.py --ask "Can a child with an egg allergy eat boiled eggs or egg pudding?" --age 5 --condition "child_above_1_year" --allergies "egg_protein" --model gemini


## Step 6.1: Diet Planning Question 1 (Gemini vs Qwen)
Testing a clinical diet planning question on both models.

In [ ]:
print('=== GEMINI RESPONSE ===')
!python main.py --ask "What is the feeding protocol for an infant with diarrhea?" --age 0.8 --weight 8.5 --condition "infant_diarrhea" --model gemini
print('\n=== QWEN LOCAL RESPONSE ===')
!python main.py --ask "What is the feeding protocol for an infant with diarrhea?" --age 0.8 --weight 8.5 --condition "infant_diarrhea" --model qwen_local

## Step 6.2: Diet Planning Question 2 (Gemini vs Qwen)
Testing an allergy-specific diet planning question on both models.

In [ ]:
print('=== GEMINI RESPONSE ===')
!python main.py --ask "Can a child with an egg allergy eat boiled eggs or egg pudding?" --age 5 --weight 18.0 --condition "child_above_1_year" --allergies "egg_protein" --model gemini
print('\n=== QWEN LOCAL RESPONSE ===')
!python main.py --ask "Can a child with an egg allergy eat boiled eggs or egg pudding?" --age 5 --weight 18.0 --condition "child_above_1_year" --allergies "egg_protein" --model qwen_local

## Step 6.3: Normal Question 1 (Gemini vs Qwen)
Testing a general nutrition knowledge question on both models.

In [ ]:
print('=== GEMINI RESPONSE ===')
!python main.py --ask "What is the expected weight doubling and tripling velocity for normal growth?" --age 1 --weight 9.5 --condition "child_above_1_year" --model gemini
print('\n=== QWEN LOCAL RESPONSE ===')
!python main.py --ask "What is the expected weight doubling and tripling velocity for normal growth?" --age 1 --weight 9.5 --condition "child_above_1_year" --model qwen_local

## Step 6.4: Normal Question 2 (Gemini vs Qwen)
Testing another general nutrition knowledge question on both models.

In [ ]:
print('=== GEMINI RESPONSE ===')
!python main.py --ask "Why is dietary fiber important in children's diet?" --age 4 --weight 15.0 --condition "child_above_1_year" --model gemini
print('\n=== QWEN LOCAL RESPONSE ===')
!python main.py --ask "Why is dietary fiber important in children's diet?" --age 4 --weight 15.0 --condition "child_above_1_year" --model qwen_local

## Step 6.5: Diet Planning Question 3 (2-Day Diet Plan)
Testing a multi-day diet plan generation question on both models.

In [ ]:
print('=== GEMINI RESPONSE ===')
!python main.py --ask "Can you give me a 2-day diet plan for my 6-year-old child who needs healthy growth and has a peanut allergy?" --age 6 --weight 20.0 --condition "healthy_growth" --allergies "nut_allergy" --model gemini
print('\n=== QWEN LOCAL RESPONSE ===')
!python main.py --ask "Can you give me a 2-day diet plan for my 6-year-old child who needs healthy growth and has a peanut allergy?" --age 6 --weight 20.0 --condition "healthy_growth" --allergies "nut_allergy" --model qwen_local

## Step 7: Execute Model Comparison Benchmark
Run evaluations over 100 questions from the dataset to compare metrics between Gemini and local Qwen2.5-7B-Instruct using actual model inference.

In [ ]:
!python main.py --evaluate --num-samples 100 --models gemini,qwen_local

## Step 8: Plot Comparison Report
Load the generated benchmark CSV and plot the results.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

report_path = os.path.join("reports", "final_model_comparison.csv")
if os.path.exists(report_path):
    df = pd.read_csv(report_path)
    
    # Convert percentage string to float for plotting
    df["Hallucination Rate (float)"] = df["Hallucination Rate"].str.rstrip('%').astype('float') / 100.0
    
    # Display the table
    print("=== Model Comparison Summary ===")
    display(df)
    
    # Plot Metrics comparison
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    # RAGAS metrics
    df.plot(x="Model", y=["Context Precision", "Context Recall", "Faithfulness", "Answer Relevancy"], kind="bar", ax=ax[0])
    ax[0].set_title("RAGAS Evaluation Scores (Higher is Better)")
    ax[0].set_ylabel("Score")
    ax[0].legend(loc="lower right")
    ax[0].grid(axis="y", linestyle="--", alpha=0.7)
    
    # Safety & Hallucination metrics
    df.plot(x="Model", y=["Safety F1", "Hallucination Rate (float)"], kind="bar", ax=ax[1], color=["green", "red"])
    ax[1].set_title("Safety F1 vs Hallucination Rate")
    ax[1].set_ylabel("Percentage / Ratio")
    ax[1].grid(axis="y", linestyle="--", alpha=0.7)
    
    plt.tight_layout()
    plt.show()
else:
    print("Error: Run Step 7 to generate the report before plotting!")